# Group 5 — Soft Robotic Arm: Model-Predictive Control on Nikoletta's One-Mode Model

Built on the course template (`soft_robotic_arm_v2.ipynb`). The assessment interface is unchanged:
the **controller cell** (§5) defines `student_controller(reference, obs)` and `reset_controller()`, and §6 runs
the instructor's `evaluate_controller` exactly as given.

**Controller.** The same model-predictive controller (MPC) as our other submissions, built on **Nikoletta's one-mode
model**: one bending mode per plane axis, $\ddot y + c\dot y + ky = b\,\Delta p$, with $k$, $c$, $b$ derived from the
arm's structure (joint springs, damping, lever arms), not fitted to data. Every 10 ms the MPC reads the supplied
trajectory one second ahead, `reference(t + 0.01 … t + 1.00)`, and computes the pressures that make the predicted tip
follow it, in closed form. Opposing segments act as pairs (S1/S3 for x, S2/S4 for y) around a 4.5 psi bias.

**Result on the instructor's example test** (§6, `make_sim(seed=2)`, 20 s, transient included): see the printed
RMSE; on our runs **0.95 mm RMSE**, against **1.26 mm** for the template's PID example. §8 compares the two on twelve
trajectories; §10 shows what changes when her model gets the gravity stiffness of the hanging arm and the measured
pneumatic lag.

Run all cells in order (Runtime → Run all), about 6 minutes on Colab. §9 (video) is optional and off by default.

## 1 — Install the arm library

Run this once in a new Google Colab runtime. It installs [soft-robotic-arm](https://pypi.org/project/soft-robotic-arm/), a
ready-to-use MuJoCo simulation of the four-segment pneumatic arm. The package
includes the configured robot model and its calibration data, so you do not
need to download the repository, open the model files, or build the robot
yourself. MuJoCo, NumPy, Matplotlib, and the video tools are installed with it.

After installation, you use the package through a small public interface:

```python
from soft_robotic_arm import make_sim

sim = make_sim(control_hz=100, seed=0)
obs = sim.reset()
obs = sim.step(command)
```

- `make_sim(...)` creates a correctly configured arm simulation.
- `reset()` returns the arm to its initial state and gives you its first sensor reading.
- `step(command)` applies one pressure command and advances the simulation by one
  control interval.
- `command` may contain four segment pressures with shape `(4,)`, or twenty
  individual pouch pressures with shape `(4, 5)`.
- The returned `obs` dictionary contains the tip position and velocity, simulation
  time, and measured segment and pouch pressures.

The install command downloads the published package and all required dependencies
directly from PyPI. If an older version was already loaded in Colab, the setup
cell reloads the installed package before creating the simulator.


**Group 5 addition:** the install line also installs `scipy`, which the controller cell uses (`scipy.linalg.expm`,
to discretise the model). Colab ships with it; the explicit install makes the notebook run elsewhere too.

In [ ]:
%pip install -q --upgrade --no-cache-dir soft-robotic-arm scipy


## 2 — Simulator setup (template)

In [ ]:
import os
import sys
from importlib.metadata import version

# Google Colab is Linux. Set the renderer before importing MuJoCo.
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# Colab can retain an already-imported package after %pip upgrades it.
# Remove only this package from the import cache, then load the installed copy.
for module_name in tuple(sys.modules):
    if module_name == "soft_robotic_arm" or module_name.startswith("soft_robotic_arm."):
        del sys.modules[module_name]

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import soft_robotic_arm
from IPython.display import Video, display
from soft_robotic_arm import make_sim

installed_version = version("soft-robotic-arm")
CONTROL_HZ = 100.0
MAX_PRESSURE_PSI = 9.0

sim = make_sim(control_hz=CONTROL_HZ, seed=0)
obs = sim.reset()
print(f"Observations: {obs}")
print(f"soft-robotic-arm version: {installed_version}")
print("Simulator ready")
print("Command rows: S1, S2, S3, S4; columns: P1, P2, P3, P4, P5")
print("Tip position [m]:", np.round(obs["tip_pos"], 4))
print("Pouch-pressure array shape:", obs["pouch_pressures"].shape)


from IPython.display import Markdown

## 3 — The model: Nikoletta's one-mode model

Derived by Nikoletta from the MuJoCo arm (her notebook `SMC_Controller_withoutdisturbanceobserver.ipynb`, section 7):
the arm's bending joints are reduced to their dominant mode. Per plane axis $i \in \{x, y\}$:
$$\ddot y_i + c\,\dot y_i + k\,y_i = b\,\Delta p_i, \qquad \Delta p_x = p_{S1} - p_{S3}, \quad \Delta p_y = p_{S2} - p_{S4}$$
- $k$ is the mode's stiffness, from the joint springs; $c$ its damping, from the joint damping; $b$ the pressure gain
  through the lever arms. Derived, not identified.
- $\Delta p$ is the commanded pressure difference: the model has **no actuator dynamics** (the pouches are assumed to
  reach their commanded pressure at once) and no z.
- On `make_sim` all four segments are active, so x is driven by S1 − S3 and y by S2 − S4.

The cell below checks the model **free-running** against `make_sim` (random steps, seed 32, and a 0.02 → 1 Hz chirp),
as written and with two additions that are *not* in her notebook: the gravity stiffness of the hanging chain (same
derivation, $k$ from the springs plus gravity) and the pneumatic lag $\tau\dot p = -p + u$ with $\tau = 0.6$ s (the
value in the simulator's configuration, `make_sim().cfg.tau_pneumatic`, and measured in our other notebooks).

In [ ]:
DT = 0.01
ONE_MODE = {"as written": (10.602415303139802, 9.230907320069726, 0.08203667215669287),
            "+ gravity stiffness": (51.45086866273482, 9.408628750381794, 0.07903934655742169)}   # k, c, |b| (m/psi)


def random_steps(seed, seconds=60.0):
    """Each segment jumps to an independent random level in 0-9 psi, held 0.5-3 s."""
    g = np.random.default_rng(seed)
    u, k = np.empty((int(seconds / DT), 4)), 0
    while k < len(u):
        n = int(g.uniform(0.5, 3.0) / DT)
        u[k:k + n] = g.uniform(0.0, 9.0, 4)
        k += n
    return u


def chirp(seconds=60.0):
    t = np.arange(int(seconds / DT)) * DT
    phase = 2 * np.pi * np.cumsum(0.02 * (1.0 / 0.02) ** (t / seconds)) * DT
    return 4.5 + 4.0 * np.column_stack([np.sin(phase), np.cos(phase), -np.sin(phase), -np.cos(phase)])


def run_open_loop(u, seed):
    """Tip x-y (mm) after each command step."""
    s = make_sim(control_hz=CONTROL_HZ, seed=seed)
    obs, tips = s.reset(), []
    for c in u:
        obs = s.step(c)
        tips.append(1000.0 * obs["tip_pos"][:2])
    s.close()
    return np.array(tips)


def free_run(u, k, c, b, lag):
    """One-mode model per axis from rest; dp = commanded (lag=False) or lagged (tau = 0.6 s) pressure difference."""
    from scipy.linalg import expm
    p = u.copy()
    if lag:
        a, prev = np.exp(-DT / 0.6), np.zeros(4)
        for i, c_ in enumerate(u):
            prev = a * prev + (1 - a) * c_
            p[i] = prev
    dp = np.column_stack([p[:, 0] - p[:, 2], p[:, 1] - p[:, 3]])
    E = expm(np.block([[np.array([[0.0, 1.0], [-k, -c]]), np.array([[0.0], [1000.0 * b]])], [np.zeros((1, 3))]]) * DT)
    A, B = E[:2, :2], E[:2, 2]
    out = np.zeros((len(u), 2))
    for i in range(2):
        x = np.zeros(2)
        for n, v in enumerate(dp[:, i]):
            x = A @ x + B * v
            out[n, i] = x[0]
    return out


rows = []
for name, u in (("random steps, seed 32", random_steps(32)), ("chirp 0.02 -> 1 Hz", chirp())):
    tips_v = run_open_loop(u, seed=3)
    for label, (k, c, b) in ONE_MODE.items():
        for lag in (False, True):
            e = free_run(u, k, c, b, lag) - tips_v
            rows.append(f"| {name} | {label}{' + pneumatic lag' if lag else ''} | {1000 * b / k:.2f} | "
                        f"{np.sqrt(np.mean(e[:, 0] ** 2)):.3f} | {np.sqrt(np.mean(e[:, 1] ** 2)):.3f} |")
display(Markdown("**Free-run check against make_sim** (RMSE, mm; the simulator's static gain is 1.81 mm per psi)\n\n"
                 "| Data | Model | static gain (mm/psi) | x | y |\n|---|---|---|---|---|\n" + "\n".join(rows)))

## 4 — The controller: MPC with trajectory preview

**State and input.** $X = (y, \dot y) \in \mathbb R^4$: tip x-y (mm, measured) and tip velocity (backward difference of
the measured tip). The input is a pair of differential corrections $\delta = (\delta_x, \delta_y)$: segment commands
$u = 4.5 + (\delta_x, \delta_y, -\delta_x, -\delta_y)$, so $\Delta p_x = 2\delta_x$ and $\Delta p_y = 2\delta_y$. The model of
§3 (as written: no actuator dynamics) is discretised with a zero-order hold at 10 ms: $X_{k+1} = AX_k + B\delta_k$,
tip $y = CX$.

**Offset.** The model is not exact. $e_k = y_k - C(AX_{k-1} + B\delta_{k-1})$ is its one-step prediction error;
its low-pass (0.5 s) is an offset $w$ added to every prediction:
$w_k = w_{k-1} + \alpha(e_k - w_{k-1})$, $\alpha = 1 - e^{-0.01/0.5}$.

**Optimisation.** Choose the next $H = 100$ corrections (1 s) to minimise
$$J = \sum_{i=0}^{H-1}\big\|C\hat X_{k+1+i} + w - r(t_k + (i+1)\,\Delta t)\big\|^2 + R_u\sum_{i=0}^{H-1}\big\|\delta_{k+i}-\delta_{k+i-1}\big\|^2$$
With $\hat Y = \Phi X_k + \Gamma U$ ($\Phi_i = CA^{i+1}$, $\Gamma_{ij} = CA^{i-j}B$ for $j \le i$, $D$ the
first-difference matrix), setting $\partial J/\partial U = 0$:
$$U^\star = \big(\Gamma^\top\Gamma + R_u D^\top D\big)^{-1}\Big[\Gamma^\top\big(\mathcal R - \Phi X_k - \mathbf 1\otimes w\big) + R_u D^\top e_1\,\delta_{k-1}\Big]$$
Only the first correction is applied: $\delta_k = g_r\,\mathcal R + g_x X_k + g_w\,w + g_u\,\delta_{k-1}$, then
$u = \operatorname{clip}_{[0,9]}\big(4.5 + (\delta_x, \delta_y, -\delta_x, -\delta_y)\big)$, sent to all five pouches of each
segment. $R_u = 1$ mm²/psi² is tuned in §7 on trajectories not used for testing.

The class also takes `gravity=True` (her derivation with the hanging chain's gravity stiffness) and
`pneumatic_lag=True` (adds the four measured segment pressures to the state, with $\tau\dot p = -p + u$); §10 uses
them. The submitted controller uses her model as written.

**Memory.** The controller keeps its last prediction, offset, previous correction and previous tip;
`reset_controller()` clears them before each trial.

## 5 — Controller cell

Self-contained: it runs on its own after the setup cell.

In [ ]:
# STUDENT CONTROLLER CELL: MPC with trajectory preview on Nikoletta's one-mode model (Group 5).
import numpy as np
from scipy.linalg import expm

# Nikoletta's one-mode model per plane axis: y'' + c y' + k y = b dp  (y in m, dp in psi), derived from the arm.
ONE_MODE_PARAMS = {False: dict(k=10.602415303139802, c=9.230907320069726, b=0.08203667215669287),   # as written
                   True: dict(k=51.45086866273482, c=9.408628750381794, b=0.07903934655742169)}    # + gravity
BIAS_PSI, LIMIT_PSI, DT_S, TAU_S = 4.5, 9.0, 0.01, 0.6
PAIRS = np.array([[1, 0], [0, 1], [-1, 0], [0, -1]], float)   # u = BIAS + PAIRS @ (dx, dy)


class PreviewMPC:
    """Unconstrained MPC on the one-mode model, reading the supplied trajectory HORIZON steps ahead."""

    def __init__(self, horizon=100, r_u=1.0, offset_tau_s=0.5, gravity=False, pneumatic_lag=False):
        prm = ONE_MODE_PARAMS[gravity]
        k, c, b = prm["k"], prm["c"], 1000.0 * prm["b"]          # b in mm per psi of dp
        O, I = np.zeros, np.eye(2)
        if pneumatic_lag:   # state (y, y', p - BIAS): dp = (p1 - p3, p2 - p4), tau p' = -p + u
            Hp = b * np.array([[1, 0, -1, 0], [0, 1, 0, -1]], float)
            Ac = np.block([[O((2, 2)), I, O((2, 4))], [-k * I, -c * I, Hp], [O((4, 4)), -np.eye(4) / TAU_S]])
            Bc = np.vstack([O((4, 4)), np.eye(4) / TAU_S]) @ PAIRS
        else:               # as written: dp = commanded difference = 2 * delta, no actuator dynamics
            Ac = np.block([[O((2, 2)), I], [-k * I, -c * I]])
            Bc = np.vstack([O((2, 2)), 2 * b * I])
        n, m, h = Ac.shape[0], 2, horizon
        self.lag = pneumatic_lag
        E = expm(np.block([[Ac, Bc], [O((m, n + m))]]) * DT_S)
        self.A, self.B = E[:n, :n], E[:n, n:]
        self.C = np.hstack([I, O((2, n - 2))])
        powers = [np.eye(n)]
        for _ in range(h):
            powers.append(self.A @ powers[-1])
        Phi = np.vstack([self.C @ powers[i + 1] for i in range(h)])
        Gam = np.zeros((2 * h, m * h))
        for i in range(h):
            for j in range(i + 1):
                Gam[2 * i:2 * i + 2, m * j:m * j + m] = self.C @ powers[i - j] @ self.B
        D = np.eye(m * h) - np.eye(m * h, k=-m)
        K = np.linalg.solve(Gam.T @ Gam + r_u * D.T @ D, np.eye(m * h))[:m]   # first move only
        self.g_r = K @ Gam.T
        self.g_x = -self.g_r @ Phi
        self.g_w = -self.g_r @ np.tile(np.eye(2), (h, 1))
        E1 = np.zeros((m * h, m))
        E1[:m] = np.eye(m)
        self.g_u = r_u * K @ D.T @ E1
        self.steps_ahead = (np.arange(h) + 1) * DT_S
        self.alpha = 1.0 - np.exp(-DT_S / offset_tau_s)
        self.reset()

    def reset(self):
        self.tip_prev, self.x_pred = None, None
        self.offset, self.delta_prev = np.zeros(2), np.zeros(2)

    def __call__(self, reference, obs):
        t = float(obs["time"])
        tip = 1000.0 * np.asarray(obs["tip_pos"], float)[:2]                     # mm
        vel = np.zeros(2) if self.tip_prev is None else (tip - self.tip_prev) / DT_S
        self.tip_prev = tip
        parts = [tip, vel] + ([np.asarray(obs["segment_pressures"], float) - BIAS_PSI] if self.lag else [])
        X = np.concatenate(parts)
        if self.x_pred is not None:                                              # model offset
            self.offset += self.alpha * ((tip - self.C @ self.x_pred) - self.offset)
        window = 1000.0 * np.array([reference(t + s) for s in self.steps_ahead])  # mm, (H, 2)
        delta = (self.g_r @ window.reshape(-1) + self.g_x @ X
                 + self.g_w @ self.offset + self.g_u @ self.delta_prev)
        segments = np.clip(BIAS_PSI + PAIRS @ delta, 0.0, LIMIT_PSI)
        applied = np.array([segments[0] - segments[2], segments[1] - segments[3]]) / 2.0
        self.x_pred = self.A @ X + self.B @ applied
        self.delta_prev = applied
        return np.repeat(segments[:, None], 5, axis=1)                          # (4, 5) pouch psi


_mpc = PreviewMPC()   # Nikoletta's model as written


def reset_controller():
    """Clear all controller memory before a new trial."""
    _mpc.reset()


def student_controller(reference, obs):
    """Track the supplied x-y trajectory; return absolute (4, 5) pouch pressures in psi."""
    return _mpc(reference, obs)


reset_controller()

## 6 — Instructor evaluation (template, unchanged)


Submit your completed **`.ipynb` notebook built on this template**, including
controller design, helper code, and any required setup. In a clearly labelled
controller cell, define **`student_controller(reference, obs)`** as shown below.
You may implement your own functions or classes and call them through this function.
This is the assessment interface; the earlier `compute(t, obs)` class is a tutorial
example. Before submitting, restart the runtime and check that your setup and
controller-definition cells run in order.

- **`reference` is the desired trajectory supplied by the instructor.** Calling
  `reference(t)` returns the absolute desired tip `[x, y]` position in metres,
  with shape `(2,)`, at simulation time `t` in seconds. The coordinates use the
  same frame as `obs["tip_pos"]`. Use the passed trajectory in your controller.
- **`obs` contains current sensor readings** from Section 3. Use `obs["time"]`
  for the current trajectory time and tip/pressure measurements for feedback.
- **Return one finite `(4, 5)` array of absolute pressure commands in psi**, with
  every value between **0 and 9**. Rows are S1–S4 and columns are P1–P5. Your
  function is called at **100 Hz**, once every 0.01 seconds.

Keep simulator creation/stepping, test-trajectory generation, and scoring outside
your controller. If the controller retains state, such as a PID integral, also
provide `reset_controller()` to clear that state before each trial. The instructor
will pass it to the evaluator as `reset=reset_controller`.

The PID example below uses stronger gains tuned in this simulator, measured tip
velocity, and a 0.02-second filter on velocity error. It estimates desired velocity
from successive reference samples. Its integral term is stored in psi and stops
accumulating when it would push a pressure correction further beyond its limit;
it can still unwind in the opposite direction. The reset function clears all
stored memory before each trial. These are example gains for this arm model,
not a guarantee of the same performance on other plants or trajectories.
For background, see [PID anti-windup and conditional integration](https://www.mathworks.com/help/simulink/slref/anti-windup-control-using-a-pid-controller.html).

**How we will test your notebook.** The instructor runs your setup and controller
cells, then pastes an evaluation function and an instructor-selected reference
trajectory into your notebook. The evaluation call is
`evaluate_controller(student_controller, instructor_reference, reset=reset_controller)`
for the PID example. The example cells below demonstrate this workflow.

Each test starts with a fresh simulator and controller state and uses the same
package version, simulator settings, initial conditions, reference, and seed for
every submission. The evaluator checks each command before applying it, compares
reference and measured tip position at matching times, and reports **planar RMSE**
and **maximum error** in millimetres, including the initial tracking transient.
Invalid shape, nonfinite values, or pressure-limit violations fail the trial.
The instructor can change the trajectory while keeping the controller interface
and testing procedure the same.


In [ ]:
# INSTRUCTOR EVALUATION CELL: an example of the function pasted into a submission.
import numpy as np
from soft_robotic_arm import make_sim


def evaluate_controller(controller, reference, *, reset=None, duration_s=20.0, seed=2):
    """Test a notebook-defined controller against any supplied trajectory.

    For a stateful controller, pass reset=reset_controller to clear its memory.
    Reference time equals obs['time']; each fresh simulation starts at zero.
    """
    if not np.isfinite(duration_s) or duration_s < 0.01:
        raise ValueError("duration_s must be finite and at least 0.01 seconds")
    if reset is not None:
        reset()

    def supplied_reference(t):
        target = np.array(reference(t), dtype=float, copy=True)
        if target.shape != (2,) or not np.all(np.isfinite(target)):
            raise ValueError("reference(t) must return a finite [x, y] vector in metres")
        return target

    test_sim = make_sim(control_hz=100.0, seed=seed)
    try:
        obs = test_sim.reset()
        times, targets, tips, commands = [], [], [], []
        for _ in range(int(round(duration_s * 100.0))):
            # Copies keep the evaluator's measurements separate from student code.
            controller_obs = {
                key: obs[key].copy() if isinstance(obs[key], np.ndarray) else obs[key]
                for key in ("time", "tip_pos", "tip_vel", "pouch_pressures", "segment_pressures")
            }
            command = np.array(
                controller(supplied_reference, controller_obs), dtype=float, copy=True
            )
            if command.shape != (4, 5) or not np.all(np.isfinite(command)):
                raise ValueError("Return a finite (4, 5) pressure array")
            if np.any(command < 0.0) or np.any(command > 9.0):
                raise ValueError("Every pressure command must be between 0 and 9 psi")

            obs = test_sim.step(command)
            # Score the new measurement and target at the same timestamp.
            times.append(obs["time"])
            targets.append(supplied_reference(obs["time"]))
            tips.append(obs["tip_pos"][:2].copy())
            commands.append(command)

        targets, tips = np.asarray(targets), np.asarray(tips)
        errors_m = np.linalg.norm(targets - tips, axis=1)
        return {
            "rmse_mm": float(1000 * np.sqrt(np.mean(errors_m**2))),
            "max_error_mm": float(1000 * np.max(errors_m)),
            "times": np.asarray(times), "references": targets,
            "tips": tips, "commands": np.asarray(commands),
        }
    finally:
        test_sim.close()


In [ ]:
# INSTRUCTOR REFERENCE AND TEST CALL: replace this with the assessment trajectory.
def instructor_reference(t):
    """Desired absolute tip [x, y] position [m] at time t [s]."""
    return 0.010 * np.array([np.sin(0.6 * t), np.sin(0.9 * t)])


# Clear PID memory before every test, including repeated evaluations.
results = evaluate_controller(
    student_controller, instructor_reference, reset=reset_controller
)
print(f"Planar RMSE: {results['rmse_mm']:.2f} mm")
print(f"Maximum error: {results['max_error_mm']:.2f} mm")


### Plot the controller evaluation

Run this cell after the evaluation call. It uses the recorded `results` to show
the desired and measured tip paths, planar tracking error, and all five pouch
pressure commands for each segment. Pressure commands are shown over the control
intervals when they were applied. In this simple PID example, the five traces in
each segment overlap because all five pouches receive the same command.


In [ ]:
# Plot the results of the most recent instructor evaluation.
import numpy as np
import matplotlib.pyplot as plt

times = results["times"]
references_mm = 1000.0 * results["references"]
tips_mm = 1000.0 * results["tips"]
errors_mm = np.linalg.norm(references_mm - tips_mm, axis=1)

performance_fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
axes = axes.ravel()
performance_fig.suptitle(
    f"Controller performance | Planar RMSE: {results['rmse_mm']:.2f} mm"
    f" | Maximum error: {results['max_error_mm']:.2f} mm",
    fontsize=14,
)

axes[0].plot(references_mm[:, 0], references_mm[:, 1], "--",
             color="0.35", linewidth=2, label="Desired trajectory")
axes[0].plot(tips_mm[:, 0], tips_mm[:, 1],
             color="tab:blue", linewidth=1.7, label="Measured tip")
axes[0].set(xlabel="x [mm]", ylabel="y [mm]", title="Desired and measured tip paths")
axes[0].set_aspect("equal", adjustable="box")
axes[0].legend()

axes[1].plot(times, errors_mm, color="tab:orange", linewidth=1.5)
axes[1].axhline(results["rmse_mm"], color="0.35", linestyle="--", label="Planar RMSE")
axes[1].set(xlabel="Time [s]", ylabel="Planar error [mm]", title="Tracking error",
            xlim=(0.0, times[-1]), ylim=(0.0, None))
axes[1].legend()

# A logged command is applied BEFORE its recorded observation timestamp.
# At 100 Hz, these edges show all held commands from t=0 to the final sample.
command_edges = np.r_[times[0] - 0.01, times]
for segment in range(4):
    ax = axes[segment + 2]
    for pouch in range(5):
        ax.stairs(results["commands"][:, segment, pouch], command_edges,
                  baseline=None, linewidth=1.2, alpha=0.85, label=f"P{pouch + 1}")
    ax.set(xlabel="Time [s]", ylabel="Command [psi]",
           title=f"S{segment + 1} pouch-pressure commands",
           xlim=(command_edges[0], times[-1]), ylim=(-0.2, 9.2), yticks=[0, 3, 6, 9])
    ax.legend(ncol=5, fontsize=8, loc="upper right")

for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.show()


## 7 — Tuning $R_u$ on separate trajectories

$R_u$ trades tracking against smooth commands. It is chosen on three trajectories that are **not** used for
testing: the template's tutorial family with seed 11 (12 mm), a smooth random path (seed 12) and random waypoints
(seed 13). Score: mean RMSE over the three with the instructor's `evaluate_controller`. The smallest mean is at
$R_u = 1$, the value in the controller cell. Our identified models tune to much smaller $R_u$; this model's static
gain is about 4× the simulator's (§3), and the larger $R_u$ damps the commands it produces.

In [ ]:
def tutorial_sines(seed, span=0.012, f=(0.05, 0.09, 0.14)):
    """The template's RandomTrajectoryPDController reference family (sum of three sines), centred at 0."""
    g = np.random.default_rng(seed)
    f, a = np.array(f), g.normal(size=(2, 3))
    a = span * a / np.abs(a).sum(1, keepdims=True)
    ph = g.uniform(0, 2 * np.pi, (2, 3))
    return lambda t: (a * np.sin(2 * np.pi * f * t + ph)).sum(1)


def smooth_random(seed, span=0.010, lo=0.03, hi=0.3):
    """Per axis six random sines (0.03-0.3 Hz), scaled to +-span."""
    g = np.random.default_rng(seed)
    f, ph, a = g.uniform(lo, hi, (2, 6)), g.uniform(0, 2 * np.pi, (2, 6)), g.uniform(0.3, 1, (2, 6))
    raw = lambda t: (a * np.sin(2 * np.pi * f * t + ph)).sum(1)
    scale = span / np.abs(np.array([raw(t) for t in np.arange(0, 20, 0.05)])).max(0)
    return lambda t: raw(t) * scale


def waypoints(seed, span=0.010, lo=1.0, hi=2.5):
    """Straight lines between random points in +-span, each 1-2.5 s; starts at the origin."""
    g = np.random.default_rng(seed)
    T, P = [0.0], [np.zeros(2)]
    while T[-1] < 25:
        T.append(T[-1] + g.uniform(lo, hi))
        P.append(g.uniform(-span, span, 2))
    P = np.array(P)
    return lambda t: np.array([np.interp(t, T, P[:, 0]), np.interp(t, T, P[:, 1])])


def circle(radius, freq):
    return lambda t: radius * np.array([np.cos(2 * np.pi * freq * t), np.sin(2 * np.pi * freq * t)])


def polygon(n, radius, lap_s):
    th = 2 * np.pi * np.arange(n + 1) / n
    v = radius * np.column_stack([np.cos(th), np.sin(th)])

    def fn(t):
        s = (t / lap_s % 1.0) * n
        k = int(s)
        return v[k] + (s - k) * (v[k + 1] - v[k])
    return fn


def saturation_pct(result):
    c = result["commands"]
    return 100.0 * np.mean((c <= 0.0) | (c >= 9.0))

In [ ]:
TUNING = {"tutorial sines, seed 11": tutorial_sines(11), "smooth random, seed 12": smooth_random(12),
          "waypoints, seed 13": waypoints(13)}
grid = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0)
rows = []
for r_u in grid:
    scores = []
    for ref in TUNING.values():
        ctl = PreviewMPC(r_u=r_u)
        scores.append(evaluate_controller(ctl, ref, reset=ctl.reset)["rmse_mm"])
    rows.append(f"| one-mode, as written (submitted) | {r_u:g} | " + " | ".join(f"{s:.3f}" for s in scores)
                + f" | **{np.mean(scores):.3f}** |")
display(Markdown("| Model | R_u | " + " | ".join(TUNING) + " | mean |\n|---|---|---|---|---|---|\n" + "\n".join(rows)))

## 8 — Comparison with the template's PID on more trajectories

The template's PID example (§7 of the template, unchanged) and our MPC, each through the instructor's
`evaluate_controller` (`make_sim(seed=2)`, 20 s, RMSE and maximum error including the initial transient). The
trajectories are absolute positions in metres, like the instructor's; some start away from the tip, which sits at
the origin at reset, so their maximum error is the unavoidable jump at $t = 0$. Saturation: % of pouch commands at
0 or 9 psi. None of these trajectories was used for tuning.

In [ ]:
# The template's PID example, unchanged, renamed so it does not replace our controller.
def reset_pid():
    global integral_psi, filtered_velocity_error, previous_reference, previous_time
    integral_psi = np.zeros(2)
    filtered_velocity_error = np.zeros(2)
    previous_reference = None
    previous_time = None


def pid_controller(reference, obs):
    global integral_psi, filtered_velocity_error, previous_reference, previous_time
    kp, ki, kd = 2000.0, 800.0, 150.0
    filter_time_s = 0.02
    bias_psi, correction_limit_psi = 4.5, 4.5
    t = float(obs["time"])
    desired_xy = np.asarray(reference(t), dtype=float)
    error = desired_xy - obs["tip_pos"][:2]
    dt = 0.0 if previous_time is None else t - previous_time
    if dt > 0.0:
        desired_velocity = (desired_xy - previous_reference) / dt
        velocity_error = desired_velocity - obs["tip_vel"][:2]
        alpha = dt / (filter_time_s + dt)
        filtered_velocity_error += alpha * (velocity_error - filtered_velocity_error)
    pd_psi = kp * error + kd * filtered_velocity_error
    candidate_integral = integral_psi.copy()
    if dt > 0.0:
        candidate_integral = np.clip(integral_psi + ki * error * dt, -correction_limit_psi, correction_limit_psi)
    trial_correction = pd_psi + candidate_integral
    pushing_limit = (((trial_correction > correction_limit_psi) & (error > 0.0))
                     | ((trial_correction < -correction_limit_psi) & (error < 0.0)))
    integral_psi = np.where(pushing_limit, integral_psi, candidate_integral)
    dx, dy = np.clip(pd_psi + integral_psi, -correction_limit_psi, correction_limit_psi)
    previous_reference = desired_xy.copy()
    previous_time = t
    segment_pressures = bias_psi + np.array([dx, dy, -dx, -dy])
    return np.repeat(segment_pressures[:, None], 5, axis=1)


TESTS = {
    "instructor example (10 mm sines)": instructor_reference,
    "tutorial sines, seed 7 (15 mm)": tutorial_sines(7, 0.015),
    "smooth random, seed 101": smooth_random(101), "smooth random, seed 102": smooth_random(102),
    "smooth random, seed 103": smooth_random(103),
    "waypoints, seed 101": waypoints(101), "waypoints, seed 102": waypoints(102), "waypoints, seed 103": waypoints(103),
    "circle 10 mm, 0.15 Hz": circle(0.010, 0.15), "triangle 10 mm, 10 s lap": polygon(3, 0.010, 10.0),
    "square 10 mm, 10 s lap": polygon(4, 0.010, 10.0), "fast circle 8 mm, 0.4 Hz": circle(0.008, 0.4),
}
compare = {}
rows = []
for name, ref in TESTS.items():
    pid = evaluate_controller(pid_controller, ref, reset=reset_pid)
    mpc = evaluate_controller(student_controller, ref, reset=reset_controller)
    compare[name] = (pid, mpc)
    rows.append(f"| {name} | {pid['rmse_mm']:.3f} | {pid['max_error_mm']:.2f} | {saturation_pct(pid):.1f} | "
                f"**{mpc['rmse_mm']:.3f}** | {mpc['max_error_mm']:.2f} | {saturation_pct(mpc):.1f} |")
display(Markdown("| Trajectory | PID RMSE | PID max | PID sat % | MPC RMSE | MPC max | MPC sat % |\n"
                 "|---|---|---|---|---|---|---|\n" + "\n".join(rows)))

In [ ]:
def overlay(name):
    pid, mpc = compare[name]
    t = mpc["times"]
    ref = 1000 * mpc["references"]
    fig, axs = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
    axs[0].plot(ref[:, 0], ref[:, 1], "--", color="0.2", lw=2.5, label="reference")
    axs[0].plot(1000 * pid["tips"][:, 0], 1000 * pid["tips"][:, 1], color="#E69F00", ls="-.", lw=2, label="PID (template)")
    axs[0].plot(1000 * mpc["tips"][:, 0], 1000 * mpc["tips"][:, 1], color="#0072B2", lw=2, label="MPC (ours)")
    axs[0].set(xlabel="x [mm]", ylabel="y [mm]", title=name)
    axs[0].set_aspect("equal", adjustable="datalim")
    for res, color, ls, lab in ((pid, "#E69F00", "-.", "PID"), (mpc, "#0072B2", "-", "MPC")):
        e = 1000 * np.linalg.norm(res["references"] - res["tips"], axis=1)
        axs[1].plot(t, e, color=color, ls=ls, lw=2, label=f"{lab}: RMSE {res['rmse_mm']:.3f} mm")
    axs[1].set(xlabel="Time [s]", ylabel="Planar error [mm]", title="Tracking error", ylim=(0, None))
    for a in axs:
        a.grid(True, alpha=0.3)
        a.legend()
    plt.show()


for name in ("instructor example (10 mm sines)", "waypoints, seed 102", "smooth random, seed 101", "square 10 mm, 10 s lap"):
    overlay(name)

## 9 — Render the controller simulation video (optional, template)

Set `RENDER_VIDEO = True` to record our controller on the instructor's example (needs a GPU-less EGL renderer, available on Colab).

In [ ]:
RENDER_VIDEO = False
if RENDER_VIDEO:
    VIDEO_RUN_TIME_S = 20.0
    VIDEO_FPS = 10
    VIDEO_PATH = "group5_mpc_tracking.mp4"
    VIDEO_STRIDE = max(1, int(round(CONTROL_HZ / VIDEO_FPS)))

    # Use a fresh simulation so video rendering is independent of the plotting run.
    video_sim = make_sim(control_hz=CONTROL_HZ, seed=2)
    reset_controller()
    video_obs = video_sim.reset()

    video_writer = imageio.get_writer(
        VIDEO_PATH, fps=VIDEO_FPS, codec="libx264", quality=7
    )
    print(f"Running {VIDEO_RUN_TIME_S:.0f} s simulation and recording {VIDEO_PATH} ...")

    for step_index in range(int(VIDEO_RUN_TIME_S * CONTROL_HZ)):
        # The controller still receives feedback on every 100 Hz control step.
        video_command = np.asarray(
            student_controller(instructor_reference, video_obs), dtype=float
        )
        if video_command.shape != (4, 5) or not np.all(np.isfinite(video_command)):
            video_writer.close()
            video_sim.close()
            raise ValueError("Controller must return a finite (4, 5) pressure matrix")

        video_obs = video_sim.step(video_command)

        # Render at 10 FPS while the controller itself continues to run at 100 Hz.
        if step_index % VIDEO_STRIDE == 0:
            frame = video_sim.render_frame(width=640, height=480)
            video_writer.append_data(frame)

    video_writer.close()
    print(f"Video saved to {VIDEO_PATH}")
    display(Video(VIDEO_PATH, embed=True, width=800, html_attributes="controls loop"))
    video_sim.close()  # Release MuJoCo's off-screen renderer.

## 10 — Variants of her model: gravity stiffness and pneumatic lag

The same MPC on four versions of the one-mode model, each with its own $R_u$ tuned on the §7 trajectories (grid
$10^{-4}$ to $10$), then run on all twelve test trajectories. *As written* is the submitted controller. *+ gravity*
uses her derivation with the hanging chain's gravity stiffness; *+ lag* adds the measured 0.6 s pneumatic lag to the
model's state. Neither addition is in her notebook. Each cell: RMSE in mm (saturation, % of pouch commands at 0 or
9 psi).

In [ ]:
VARIANTS = {"as written": dict(), "+ gravity": dict(gravity=True), "+ lag": dict(pneumatic_lag=True),
            "+ gravity + lag": dict(gravity=True, pneumatic_lag=True)}
best_r_u = {}
for name, kw in VARIANTS.items():
    means = {}
    for r_u in (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0):
        scores = []
        for ref in TUNING.values():
            ctl = PreviewMPC(r_u=r_u, **kw)
            scores.append(evaluate_controller(ctl, ref, reset=ctl.reset)["rmse_mm"])
        means[r_u] = np.mean(scores)
    best_r_u[name] = min(means, key=means.get)
print("tuned R_u:", best_r_u)

head = "| Trajectory | " + " | ".join(f"{n} (R_u {best_r_u[n]:g})" for n in VARIANTS) + " |"
rows = []
for test, ref in TESTS.items():
    cells = []
    for name, kw in VARIANTS.items():
        ctl = PreviewMPC(r_u=best_r_u[name], **kw)
        res = evaluate_controller(ctl, ref, reset=ctl.reset)
        cells.append(f"{res['rmse_mm']:.3f} ({saturation_pct(res):.0f} %)")
    rows.append(f"| {test} | " + " | ".join(cells) + " |")
display(Markdown(head + "\n|" + "---|" * (len(VARIANTS) + 1) + "\n" + "\n".join(rows)))

## 11 — Summary and limitations

- **Her model as written** already gives a working tracking controller: about 0.95 mm RMSE on the instructor's example
  against 1.26 mm for the template's PID (§6, §8). The gain comes from reading the supplied trajectory ahead.
- **Its two gaps on `make_sim`** (§3, §10): the static gain is about 4× the simulator's, because the gravity stiffness
  of the hanging arm is left out, and the commanded pressure is assumed to act at once, while the pouches lag by
  0.6 s. With both added, the same MPC reaches about 0.17 mm on the instructor's example; the same structure fitted to
  data (our M3sim notebook) reaches 0.09 mm.
- **Where all controllers are limited:** trajectories that start away from the tip carry an unavoidable initial error,
  and fast paths (0.4 Hz circle) saturate the valves.
- **Assumptions:** the reference function can be queried ahead of the current time, as the interface allows; the tip
  velocity is a raw backward difference, which real sensor noise would amplify.